# Scratch: week of Oct 19-25
Exploration notebook. Data comes from the company app (Supabase); credentials load from `.env` in the project root.

In [1]:
import sys
from datetime import timedelta
sys.path.insert(0, "../src")
from reusables import *
from supabase_pull import load_waste, count_jobs, split_baseline_events, pull
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.float_format", "{:,.2f}".format)

# ---- Parameters -------------------------------------------------------------
WEEK_START  = "2026-10-19"      # Monday of the week
WEEK_LABEL  = "Week of Oct 19-25"
WEEK_JOBS   = None              # None = count from the app
LOAD_START  = "2026-01-01"
# ---------------------------------------------------------------------------

week_start = pd.Timestamp(WEEK_START).date()
week_end   = week_start + timedelta(days=7)          # following Monday (not included)

# All waste events up to the end of the week (app timestamps are UTC, so pull one extra day and filter on local date)
df = load_waste(LOAD_START, end=str(week_end + timedelta(days=1)))
df = df[df["incident_date"] < week_end].copy()
df["kind"] = df["kind"].replace({"Lost Item": "Item Search"})

baseline, after_baseline = split_baseline_events(df)
this_week = after_baseline[after_baseline["incident_date"] >= week_start]

if WEEK_JOBS is None:                                 # jobs with labor logged in the app that week
    WEEK_JOBS = count_jobs(str(week_start), str(week_end - timedelta(days=1)))

checks = run_checks(df, {
    "missing date":   df["incident_date"].isna(),
    "zero cost":      df["total_cost"].le(0),
    "no job linked":  df["job_id"].isna() & df["kind"].isin(["Breakage", "Reclean", "Repeat Trip"]),
})
print(f"{len(df)} events loaded | baseline {len(baseline)} | {WEEK_LABEL}: {len(this_week)} events, {WEEK_JOBS} jobs")

23 events loaded | baseline 14 | Week of Oct 19-25: 9 events, 24 jobs


## Data checks

In [2]:
checks

,rows_flagged,of_total,pct
no job linked,5,23,21.70
missing date,0,23,0.00
zero cost,0,23,0.00


## This week by type

In [3]:
per_100_jobs(this_week, WEEK_JOBS)

,events,total,events_per_100,cost_per_100
kind,,,,
Breakage,6,631.50,25.00,"2,631.25"
Item Search,2,378.00,8.30,"1,575.00"
Repeat Trip,1,112.00,4.20,466.67


## Baseline vs this week

In [4]:
compare_weeks(baseline, this_week, base_jobs=36, cur_jobs=WEEK_JOBS, cur_label=WEEK_LABEL)

Baseline                           Week of Oct 19-25           \
              Events Cost ($) Per 100 jobs ($)            Events Cost ($)   
kind                                                                        
Breakage        9.00   745.00         2,069.44              6.00   631.50   
Item Search     5.00   903.00         2,508.33              2.00   378.00   
Repeat Trip     0.00     0.00             0.00              1.00   112.00   
Total          14.00 1,648.00         4,577.78              9.00 1,121.50   

                                Change         
            Per 100 jobs ($) $ per 100      %  
kind                                           
Breakage            2,631.25    561.81  27.00  
Item Search         1,575.00   -933.33 -37.00  
Repeat Trip           466.67    466.67    NaN  
Total               4,672.92     95.14   2.00

## Event detail

In [5]:
this_week[["incident_date", "event_no", "kind", "job", "job_type", "title", "hours", "labor_cost", "other_cost", "total_cost"]]

,incident_date,event_no,kind,job,job_type,title,hours,labor_cost,other_cost,total_cost
14,2026-10-19,W-2026-0015,Breakage,6260 Lake Plantation Dr,water,Furniture damaged,1.00,21.00,0.00,21.00
15,2026-10-19,W-2026-0016,Breakage,6260 Lake Plantation Dr,water,Art / decor damaged,1.00,21.00,0.00,21.00
16,2026-10-19,W-2026-0017,Breakage,898 Bent Creek Dr,fire,Personal items damaged,1.00,21.00,0.00,21.00
17,2026-10-19,W-2026-0018,Breakage,1218 Prince Rd,fire,Furniture damaged,1.50,31.50,0.00,31.50
18,2026-10-19,W-2026-0019,Breakage,2066 Holly Oaks River Dr,fire,Furniture damaged,2.00,42.00,0.00,42.00
19,2026-10-19,W-2026-0020,Item Search,NaN,NaN,Search for item (16 hr),16.00,336.00,0.00,336.00
20,2026-10-19,W-2026-0021,Item Search,NaN,NaN,Search for item (2 hr),2.00,42.00,0.00,42.00
21,2026-10-19,W-2026-0022,Repeat Trip,139 Silver Pins Dr,fire,Repeat trip – Gutzeit (70 mi),3.00,63.00,49.00,112.00
22,2026-10-22,W-2026-0023,Breakage,NaN,NaN,Furniture damaged,0.00,0.00,495.00,495.00


## Labor logged this week
`pull()` runs any read-only query and returns a DataFrame. Pass `snapshot="name"` to save a dated copy to `data/raw`.

In [6]:
labor = pull("""
    select l.work_date, l.stage, l.hours_worked, l.job_id, j.service_type
    from daily_labor l
    left join jobs j on j.id = l.job_id
    where l.work_date between %(s)s and %(e)s
""", {"s": str(week_start), "e": str(week_end - timedelta(days=1))})

labor.groupby(["work_date", "stage"])["hours_worked"].sum().unstack(fill_value=0)

stage,chemical_clean,demo,laundry,move_back,move_in,non_salvage,office,post_construction_clean,residential_clean,searching,structure_clean,warehouse,water_mitigation
work_date,,,,,,,,,,,,,
2026-10-19,0.00,0.00,0.00,39.75,28.78,0.00,10.07,0.00,37.63,0.00,13.98,0.00,7.85
2026-10-20,47.03,0.00,8.06,0.00,0.00,0.00,16.95,0.00,67.56,4.54,14.00,0.79,13.61
2026-10-21,6.79,0.00,8.34,0.00,7.54,8.77,19.77,39.00,27.33,3.68,12.80,0.00,13.74
2026-10-22,0.00,7.60,0.00,0.00,0.00,5.90,18.92,42.14,2.52,0.00,14.02,29.56,6.23
2026-10-23,0.00,0.00,8.41,48.65,0.00,0.00,8.84,39.21,0.00,0.00,12.47,0.01,3.36
2026-10-24,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.44


## Scratch